# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [5]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [6]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [7]:
# Task 1 — Multi-series line with highlight
# YOUR CODE HERE
# Let's apply the principles

a= df[df['Region'] == 'Asia']
highlight = 'China'

# Build colour map: highlight in blue, everything else grey
color_map = {c: 'DarkBlue' if c == highlight else 'LightGrey' for c in a['Country'].unique()}

fig = px.line(a, x='Year', y='CO2_Mt', color='Country',
              color_discrete_map=color_map,
              labels={'CO2_Mt': 'CO2 Emissions (Mt)', 'Year': ''})

fig.update_traces(
    line=dict(width=1.5),   # default: thin for context countries
    showlegend=False      # direct label replaces legend (Gestalt proximity)
)

# Override highlighted country: thicker line
fig.update_traces(
    line=dict(width=3),
    selector=dict(name=highlight)
)

# Direct label at end of highlighted line
last = a.loc[(a['Country'] == highlight) & (a['Year'] == a['Year'].max())]

fig.add_annotation(
    x=last['Year'].values[0], y=last['CO2_Mt'].values[0],
    text=f'<b>{highlight}</b>', showarrow=False,
    xanchor='left', xshift=6,
    font=dict(color='DarkBlue', size=12, family='Arial')
)

fig.update_layout(
    title="China's CO2 emissions tripled 2000–2022 — no other country comes close",
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    yaxis=dict(gridcolor='#EEEEEE', title='CO2 Emissions (Mt)'),
    xaxis=dict(showgrid=False, title=''),
    margin=dict(l=60, r=80, t=55, b=40)
)

fig.show()



---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [19]:
import pandas as pd
import plotly.graph_objects as go

# Load data
df = pd.read_csv('../data/co2_emissions.csv')

# Aggregate
df_region = df.groupby(['Region', 'Year'])['CO2_Mt'].mean().reset_index()
df_filtered = df_region[df_region['Year'].isin([2000, 2022])]

# Pivot
df_pivot = df_filtered.pivot(index='Region', columns='Year', values='CO2_Mt').reset_index()
df_pivot.columns = ['Region', 'CO2_2000', 'CO2_2022']

# Change
df_pivot['Change'] = df_pivot['CO2_2022'] - df_pivot['CO2_2000']

# Identify key regions
top_increase = df_pivot.sort_values('Change', ascending=False).iloc[0]
top_decrease = df_pivot.sort_values('Change').iloc[0]

# Axis shift (KEY FIX)
x_axis_shift = 0.12   # moves axis right of labels

fig = go.Figure()

for _, row in df_pivot.iterrows():
    region = row['Region']

    # Style
    if region == top_increase['Region']:
        color = '#d62728'
        width = 4
        highlight = True
    elif region == top_decrease['Region']:
        color = '#2ca02c'
        width = 4
        highlight = True
    else:
        color = '#D3D3D3'
        width = 2
        highlight = False

    # Shifted X positions
    x_vals = [x_axis_shift, 1]

    fig.add_trace(go.Scatter(
        x=x_vals,
        y=[row['CO2_2000'], row['CO2_2022']],
        mode='lines',
        line=dict(color=color, width=width),
        showlegend=False
    ))

    # LEFT labels (before axis)
    if highlight:
        fig.add_annotation(
            x=x_axis_shift - 0.02,
            y=row['CO2_2000'],
            text=f"{region} ({row['CO2_2000']:.0f})",
            showarrow=False,
            xanchor='right',
            font=dict(size=13, color=color)
        )

        # RIGHT values
        fig.add_annotation(
            x=1.03,
            y=row['CO2_2022'],
            text=f"{row['CO2_2022']:.0f}",
            showarrow=False,
            xanchor='left',
            font=dict(size=13, color=color)
        )

# Layout
fig.update_layout(
    title="Asia surges; North America declines (2000–2022)",
    width=1000,
    height=500,
    plot_bgcolor='white',

    xaxis=dict(
        tickvals=[x_axis_shift, 1],
        ticktext=['2000', '2022'],
        showline=True,
        linecolor='grey',
        linewidth=1,
        showgrid=False,
        zeroline=False
    ),

    yaxis=dict(
        showline=True,
        linecolor='grey',
        linewidth=1,
        showgrid=False,
        zeroline=False,
        showticklabels=False
    ),

    margin=dict(l=160, r=120, t=80, b=50)
)

fig.show()